Compute changes in protocol quality and criterion coverage across Delphi rounds.....


In [1]:
from pathlib import Path
import os
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(PROJECT_ROOT).startswith('\\\\?\\')):
    PROJECT_ROOT = Path('\\\\?\\' + str(PROJECT_ROOT))


In [2]:
import io
import json
import os
import re
import sys
import warnings
import numpy as np
import pandas as pd

def read_csv_normalized(*args, **kwargs):
    data = pd.read_csv(*args, **kwargs)
    if 'Vignette' in data.columns:
        data['Vignette'] = data['Vignette'].map(lambda value: value.replace('\r\n', '\n') if isinstance(value, str) else value)
    return data
from scipy import stats
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore')
ROOT = str(PROJECT_ROOT)
OUT = os.path.normpath(os.path.join(ROOT, 'analyses/round_dynamics/cohort_and_round_effects'))
os.makedirs(OUT, exist_ok=True)
VARIANTS = ['non', 'full']
ALPHA = 0.05
PBASE = {'non': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_truncated', 'full': 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'}
HOMOG_DS = {'non': 'experiments/homogeneous_panels/three_deepseekv4_members_truncated', 'full': 'experiments/homogeneous_panels/three_deepseekv4_members_complete'}
FIX_PBASE = 'experiments/fixed_length_controls/delphi_panels/internal_panel_variants/standard_protocol/heterogeneous_gpt120_organizer'
NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_noroles_%s_judge.csv'
FB = 'experiments/fixed_length_controls/single_model_baselines/baseline_%s/llm_as_judge/baseline_noroles_%s_judge.csv'

def arm_paths(v):
    P, H = (PBASE[v], HOMOG_DS[v])
    return {'nofix_original': (P + '/llm_as_judge/delphi_base_noroles_judge.csv', P + '/llm_as_judge/round_evaluation/delphi_base_noroles_protocol_r%d_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'nofix'), 'fix_original': (FIX_PBASE + '/llm_as_judge/delphi_base_noroles_judge.csv', 'experiments/fixed_length_controls/round_evaluation/delphi_base_noroles_protocol_r%d_judge.csv', ['gpt', 'qwen', 'deepseekv4'], 'fix'), 'pstrong': ('experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/llm_as_judge/delphi_pstrong_noroles_judge.csv', 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120/llm_as_judge/round_evaluation/delphi_pstrong_noroles_protocol_r%d_judge.csv', ['deepseekv4', 'glm', 'gpt'], 'nofix'), 'pweak': ('experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/llm_as_judge/delphi_pweak_noroles_judge.csv', 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro/llm_as_judge/round_evaluation/delphi_pweak_noroles_protocol_r%d_judge.csv', ['qwen', 'gemini', 'deepseekv4pro'], 'nofix'), 'pmixed': ('experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm/llm_as_judge/delphi_pmixed_noroles_judge.csv', 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm/llm_as_judge/round_evaluation/delphi_pmixed_noroles_protocol_r%d_judge.csv', ['deepseekv4', 'gemini', 'glm'], 'nofix'), 'homog_gpt': ('experiments/homogeneous_panels/three_gpt_members/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'experiments/homogeneous_panels/three_gpt_members/llm_as_judge/round_evaluation/delphi_homog_gpt_noroles_protocol_r%d_judge.csv', ['gpt'], 'nofix'), 'homog_gemini': ('experiments/homogeneous_panels/three_gemini_members/llm_as_judge/delphi_homog_gemini_noroles_judge.csv', 'experiments/homogeneous_panels/three_gemini_members/llm_as_judge/round_evaluation/delphi_homog_gemini_noroles_protocol_r%d_judge.csv', ['gemini'], 'nofix'), 'homog_deepseekv4': (H + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv', H + '/llm_as_judge/round_evaluation/delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv', ['deepseekv4'], 'nofix')}

def read(rel):
    p = os.path.normpath(os.path.join(ROOT, rel))
    if not os.path.exists(p):
        return None
    d = read_csv_normalized(p, sep=';', encoding='utf-8-sig', low_memory=False)
    if 'table_source' in d.columns:
        d = d[d['table_source'].fillna('result_df') == 'result_df']
    if 'Judge_score' not in d.columns:
        return None
    d = d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette', keep='first')
    return d.set_index('Vignette')

def panel(v, name):
    fin_p, rnd_p, members, regime = arm_paths(v)[name]
    fin, r1, r2 = (read(fin_p), read(rnd_p % 1), read(rnd_p % 2))
    if fin is None or r1 is None:
        return None
    ix = fin.index.intersection(r1.index)
    if r2 is not None:
        ix = ix.intersection(r2.index)
    t = pd.DataFrame({'FIN': fin.loc[ix, 'Judge_score'].astype(float), 'R1': r1.loc[ix, 'Judge_score'].astype(float), 'R2': r2.loc[ix, 'Judge_score'].astype(float) if r2 is not None else np.nan, 'cohort_src': r1.loc[ix, 'protocol_source'], 'rounds': pd.to_numeric(fin.loc[ix, 'delphi_rounds_executed'], errors='coerce'), 'Type': fin.loc[ix, 'Type']})
    if 'protocol_n_valid_votes' in r1.columns:
        t['n_votes_r1'] = pd.to_numeric(r1.loc[ix, 'protocol_n_valid_votes'], errors='coerce')
    t['cohort'] = t['rounds'].map({1: 'C1', 2: 'C2', 3: 'C3'})
    B = FB if regime == 'fix' else NB
    for m in members:
        b = read(B % (m, m))
        if b is not None:
            t['base_' + m] = b['Judge_score'].astype(float)
    have = ['base_' + m for m in members if 'base_' + m in t.columns]
    if have:
        t['mean_single'] = t[have].mean(axis=1)
        t['max_single'] = t[have].max(axis=1)
    return t.dropna(subset=['FIN', 'R1'])
REG = []

def log_test(family, variant, arm, contrast, n, effect, p, note=''):
    REG.append({'family': family, 'variant': variant, 'arm': arm, 'contrast': contrast, 'n': n, 'effect': effect, 'p': p, 'note': note})
    return p

def wilc(a, b):
    d = (a - b).dropna()
    nz = d[d != 0]
    p = stats.wilcoxon(d[d.notna()])[1] if len(nz) else np.nan
    return (len(d), len(nz), float(d.mean()), float((d > 0).mean()), float(p))

def mde(x, alpha=0.05, power=0.8):
    x = np.asarray(x, float)
    n = len(x)
    if n < 3:
        return np.nan
    sd = x.std(ddof=1)
    return (stats.norm.isf(alpha / 2) + stats.norm.isf(1 - power)) * sd / np.sqrt(n)

def tost(x, bound):
    x = np.asarray(x, float)
    n = len(x)
    if n < 3:
        return np.nan
    se = x.std(ddof=1) / np.sqrt(n)
    if se == 0:
        return 0.0
    df = n - 1
    p_lo = stats.t.sf((x.mean() - -bound) / se, df)
    p_hi = stats.t.cdf((x.mean() - bound) / se, df)
    return float(max(p_lo, p_hi))

def b1_cohorts(v):
    rows = []
    for name in arm_paths(v):
        t = panel(v, name)
        if t is None:
            continue
        for coh, sub in [('C1', t[t.cohort == 'C1']), ('C2', t[t.cohort == 'C2']), ('C3', t[t.cohort == 'C3']), ('C2+C3', t[t.cohort != 'C1']), ('все', t)]:
            if not len(sub):
                continue
            n, nz, d1, sh1, p1 = wilc(sub['FIN'], sub['R1'])
            r = {'плечо': name, 'когорта': coh, 'n': n, 'n≠0': nz, 'R1': sub['R1'].mean(), 'FIN': sub['FIN'].mean(), 'FIN−R1': d1, 'доля>': sh1, 'p(FIN~R1)': p1}
            if sub['R2'].notna().any():
                _, nz2, d2, sh2, p2 = wilc(sub['FIN'], sub['R2'])
                r.update({'R2': sub['R2'].mean(), 'FIN−R2': d2, 'p(FIN~R2)': p2})
            rows.append(r)
            if coh in ('C2+C3', 'C3'):
                log_test('A. когорты FIN−R1', v, name, 'FIN−R1 @' + coh, n, d1, p1)
        c3 = t[t.cohort == 'C3']
        if len(c3) > 3 and c3['R2'].notna().any():
            n, _, d, sh, p = wilc(c3['R2'], c3['R1'])
            log_test('B. шаги внутри C3', v, name, 'R1→R2 @C3', n, d, p)
            n, _, d, sh, p = wilc(c3['FIN'], c3['R2'])
            log_test('B. шаги внутри C3', v, name, 'R2→R3 @C3', n, d, p)
    return pd.DataFrame(rows)

def b2_cohort_stability():
    rows = []
    for name in ['nofix_original', 'homog_deepseekv4']:
        a, b = (panel('non', name), panel('full', name))
        ix = a.index.intersection(b.index)
        x = a.loc[ix, 'cohort'] != 'C1'
        y = b.loc[ix, 'cohort'] != 'C1'
        tab = pd.crosstab(x, y)
        n11 = int((x & y).sum())
        n10 = int((x & ~y).sum())
        n01 = int((~x & y).sum())
        n00 = int((~x & ~y).sum())
        den = np.sqrt(float((n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00)))
        phi = (n11 * n00 - n10 * n01) / den if den else np.nan
        rows.append({'плечо': name, 'n': len(ix), 'трудных non': int(x.sum()), 'трудных full': int(y.sum()), 'в обоих': n11, 'только non': n10, 'только full': n01, 'phi': phi, 'p (McNemar)': float(stats.binomtest(n10, n10 + n01, 0.5).pvalue) if n10 + n01 else np.nan})
    return pd.DataFrame(rows).set_index('плечо')

def b3_mechanism():
    rows = []
    for v in VARIANTS:
        for name in ['nofix_original', 'homog_deepseekv4', 'homog_gpt', 'pmixed']:
            t = panel(v, name)
            if t is None or 'n_votes_r1' not in t.columns:
                continue
            nv = t['n_votes_r1']
            full_v = nv >= 3
            r = {'вариант': v, 'плечо': name, 'n': len(t), 'доля полных голосований R1': float(full_v.mean()), 'R1 при полном': float(t.loc[full_v, 'R1'].mean()) if full_v.any() else np.nan, 'R1 при неполном': float(t.loc[~full_v, 'R1'].mean()) if (~full_v).any() else np.nan}
            if full_v.any() and (~full_v).any():
                u = stats.mannwhitneyu(t.loc[full_v, 'R1'], t.loc[~full_v, 'R1'])
                r['p (MWU)'] = float(u.pvalue)
                r['доля трудных при полном'] = float((t.loc[full_v, 'cohort'] != 'C1').mean())
                r['доля трудных при неполном'] = float((t.loc[~full_v, 'cohort'] != 'C1').mean())
                log_test('C. механизм: неполнота R1', v, name, 'R1 полное vs неполное голосование', len(t), r['R1 при полном'] - r['R1 при неполном'], r['p (MWU)'])
            rows.append(r)
    return pd.DataFrame(rows).set_index(['вариант', 'плечо'])

def b4_type(v):
    rows = []
    for name in arm_paths(v):
        t = panel(v, name)
        if t is None:
            continue
        g = t[t.cohort != 'C1']
        if len(g) < 10:
            continue
        groups = []
        for ty, sub in g.groupby('Type'):
            d = sub['FIN'] - sub['R1']
            groups.append(d.values)
            rows.append({'плечо': name, 'Type': ty, 'n': len(sub), 'FIN−R1': float(d.mean()), 'доля>': float((d > 0).mean())})
            if len(sub) >= 5:
                p = stats.wilcoxon(d)[1] if (d != 0).any() else np.nan
                log_test('D. Type-подгруппы @C2+C3', v, name, 'FIN−R1 | %s' % ty, len(sub), float(d.mean()), p)
        if len(groups) >= 2:
            kw = stats.kruskal(*groups)
            rows.append({'плечо': name, 'Type': '— Kruskal-Wallis —', 'n': len(g), 'FIN−R1': np.nan, 'доля>': np.nan, 'p': float(kw.pvalue)})
            log_test('E. Kruskal-Wallis по Type', v, name, 'разброс FIN−R1 по Type', len(g), np.nan, float(kw.pvalue))
    return pd.DataFrame(rows)

def b5_regression_to_mean(v):
    rows = []
    for name in arm_paths(v):
        t = panel(v, name)
        if t is None:
            continue
        g = t[t.cohort != 'C1'].copy()
        if len(g) < 20:
            continue
        g['d'] = g['FIN'] - g['R1']
        rho, p = stats.spearmanr(g['R1'], g['d'])
        log_test('F. регрессия к середине', v, name, 'corr(R1, FIN−R1) @C2+C3', len(g), float(rho), float(p))
        q = pd.qcut(g['R1'], 4, labels=['Q1 худшие R1', 'Q2', 'Q3', 'Q4 лучшие R1'], duplicates='drop')
        for lab, sub in g.groupby(q, observed=True):
            rows.append({'плечо': name, 'квартиль': lab, 'n': len(sub), 'R1': sub['R1'].mean(), 'FIN−R1': sub['d'].mean(), 'доля>': float((sub['d'] > 0).mean())})
        rows.append({'плечо': name, 'квартиль': '— Spearman —', 'n': len(g), 'R1': np.nan, 'FIN−R1': rho, 'доля>': p})
    return pd.DataFrame(rows)

def b6_vs_single(v):
    rows = []
    for name in arm_paths(v):
        t = panel(v, name)
        if t is None or 'max_single' not in t.columns:
            continue
        for coh, sub in [('C2+C3', t[t.cohort != 'C1']), ('все', t)]:
            if len(sub) < 10:
                continue
            for stage in ['R1', 'FIN']:
                n, _, d, sh, p = wilc(sub[stage], sub['max_single'])
                rows.append({'плечо': name, 'когорта': coh, 'стадия': stage, 'n': n, 'Δ к max(single)': d, 'доля>': sh, 'p': p})
                if coh == 'C2+C3':
                    log_test('G. стадия против одиночной', v, name, '%s − max(single) @C2+C3' % stage, n, d, p)
    return pd.DataFrame(rows)

def b7_pooled(v):
    import statsmodels.formula.api as smf
    recs = []
    for name in arm_paths(v):
        t = panel(v, name)
        if t is None:
            continue
        g = t[t.cohort != 'C1']
        for stage in ['R1', 'FIN']:
            recs.append(pd.DataFrame({'case': g.index, 'arm': name, 'stage': stage, 'y': g[stage].values}))
    d = pd.concat(recs, ignore_index=True)
    d['is_fin'] = (d['stage'] == 'FIN').astype(float)
    m0 = smf.mixedlm('y ~ is_fin + C(arm)', d, groups=d['case']).fit(reml=False)
    p_main = float(m0.pvalues.get('is_fin', np.nan))
    log_test('H. объединённая модель', v, 'все 8 плеч', 'общий эффект стадии FIN vs R1 @C2+C3', len(d), float(m0.params.get('is_fin', np.nan)), p_main)
    m1 = smf.mixedlm('y ~ is_fin * C(arm)', d, groups=d['case']).fit(reml=False)
    inter = [i for i in m1.params.index if 'is_fin:' in i]
    tab = pd.DataFrame({'коэф': m1.params[inter], 'SE': m1.bse[inter], 'p': m1.pvalues[inter]})
    lr = 2 * (m1.llf - m0.llf)
    p_inter = float(stats.chi2.sf(lr, len(inter)))
    log_test('H. объединённая модель', v, 'все 8 плеч', 'взаимодействие стадия x плечо (LR-тест)', len(d), lr, p_inter)
    main = pd.DataFrame({'коэф': m0.params, 'SE': m0.bse, 'p': m0.pvalues}).drop('Group Var')
    return (main, tab, p_inter)

def b7b_mediation():
    rows = []
    for v in VARIANTS:
        for name in ['homog_deepseekv4', 'homog_gpt']:
            t = panel(v, name)
            if t is None or 'n_votes_r1' not in t.columns:
                continue
            g = t[t.cohort != 'C1'].copy()
            if len(g) < 15:
                continue
            g['d'] = g['FIN'] - g['R1']
            k = int(g['n_votes_r1'].max())
            lost = k - g['n_votes_r1']
            rho, p = stats.spearmanr(lost, g['d'])
            r = {'вариант': v, 'плечо': name, 'n': len(g), 'потеряно голосов, среднее': float(lost.mean()), 'corr(потеряно, FIN−R1)': float(rho), 'p': float(p)}
            for L, sub in g.groupby(lost.clip(upper=2)):
                r['FIN−R1 | потеряно %d' % L] = float(sub['d'].mean())
                r['n | потеряно %d' % L] = len(sub)
            rows.append(r)
            log_test('H3. медиация через голоса', v, name, 'corr(потеряно голосов, FIN−R1) @C2+C3', len(g), float(rho), float(p))
    return pd.DataFrame(rows).set_index(['вариант', 'плечо'])

def b8_axes(v, arms=('homog_deepseekv4', 'nofix_original')):

    def axis_frame(rel):
        d = read(rel)
        if d is None:
            return None
        recs = {}
        for vig, s in d['judge_details'].dropna().items():
            try:
                items = json.loads(s)
            except Exception:
                continue
            acc = {}
            for it in items:
                ax = next((x.split(':', 1)[1] for x in it.get('tags', []) if isinstance(x, str) and x.startswith('axis:')), None)
                if ax is None:
                    continue
                pts = it.get('points', 0) or 0
                e, m = acc.get(ax, (0.0, 0.0))
                acc[ax] = (e + (pts if it.get('met') else 0.0), m + (pts if pts > 0 else 0.0))
            recs[vig] = {k: e / m if m else np.nan for k, (e, m) in acc.items()}
        return pd.DataFrame(recs).T
    rows = []
    for name in arms:
        fin_p, rnd_p, _, _ = arm_paths(v)[name]
        F, R = (axis_frame(fin_p), axis_frame(rnd_p % 1))
        if F is None or R is None:
            continue
        t = panel(v, name)
        gen = t[t.cohort != 'C1'].index
        for ax in ['reasoning', 'accuracy', 'completeness', 'urgency', 'uncertainty', 'communication_quality']:
            if ax not in F.columns or ax not in R.columns:
                continue
            ix = F.index.intersection(R.index).intersection(gen)
            a, b = (F.loc[ix, ax].astype(float), R.loc[ix, ax].astype(float))
            ok = a.notna() & b.notna()
            n, _, d, sh, p = wilc(a[ok], b[ok])
            rows.append({'плечо': name, 'ось': ax, 'n': n, 'R1': b[ok].mean(), 'FIN': a[ok].mean(), 'FIN−R1': d, 'доля>': sh, 'p': p})
            log_test('I. оси рубрики @C2+C3', v, name, 'FIN−R1 | ось %s' % ax, n, d, p)
    return pd.DataFrame(rows)

def b9_equivalence(v):
    rows = []
    for name in arm_paths(v):
        t = panel(v, name)
        if t is None:
            continue
        g = t[t.cohort != 'C1']
        if len(g) < 10:
            continue
        d = (g['FIN'] - g['R1']).values
        rows.append({'плечо': name, 'n': len(d), 'FIN−R1': float(d.mean()), 'p (Уилкоксон)': wilc(g['FIN'], g['R1'])[4], 'TOST ±0.05': tost(d, 0.05), 'TOST ±0.10': tost(d, 0.1), 'MDE (80%)': mde(d)})
    return pd.DataFrame(rows).set_index('плечо')

def b10_multiplicity():
    r = pd.DataFrame(REG).dropna(subset=['p'])
    out = []
    for (fam, v), sub in r.groupby(['family', 'variant']):
        h = multipletests(sub['p'], alpha=ALPHA, method='holm')[0]
        b = multipletests(sub['p'], alpha=ALPHA, method='fdr_bh')[0]
        out.append({'семейство': fam, 'вариант': v, 'тестов': len(sub), 'p<0.05 сырых': int((sub['p'] < ALPHA).sum()), 'Holm': int(h.sum()), 'BH': int(b.sum())})
    per_fam = pd.DataFrame(out).set_index(['семейство', 'вариант']).sort_index()
    tot = []
    for v, sub in r.groupby('variant'):
        h = multipletests(sub['p'], alpha=ALPHA, method='holm')[0]
        b = multipletests(sub['p'], alpha=ALPHA, method='fdr_bh')[0]
        tot.append({'вариант': v, 'тестов всего': len(sub), 'p<0.05 сырых': int((sub['p'] < ALPHA).sum()), 'Holm': int(h.sum()), 'BH': int(b.sum()), 'ожидалось ложных при H0': round(ALPHA * len(sub), 1)})
    return (per_fam, pd.DataFrame(tot).set_index('вариант'), r)

def show(title, obj, note=''):
    print('\n' + '=' * 104)
    print(title)
    if note:
        print(note)
    print('=' * 104)
    print('(нет данных)' if obj is None else obj.to_string() if isinstance(obj, (pd.DataFrame, pd.Series)) else str(obj))

def main():
    print('=' * 104)
    print('УГЛУБЛЁННАЯ ПРОВЕРКА ЭФФЕКТА РАУНДОВ + ИНВЕНТАРИЗАЦИЯ ЗНАЧИМОСТИ')
    print('=' * 104)
    print(__doc__.split('Запуск:')[0].strip())
    for v in VARIANTS:
        t = b1_cohorts(v)
        show('A. КОГОРТНЫЙ ДИЗАЙН (воспроизведение analysis_rounds_vs_final §2) — %s' % v.upper(), t.set_index(['плечо', 'когорта']), 'n≠0 -- сколько кейсов дали НЕнулевую разность; у C1 их ноль по построению.')
        t.to_csv(os.path.normpath(os.path.join(OUT, {'non': 'round_cohorts_truncated_runs.csv', 'full': 'round_cohorts_complete_runs.csv'}[v])), index=False, encoding='utf-8-sig')
    show('B. УСТОЙЧИВОСТЬ СОСТАВА «ТРУДНОЙ» КОГОРТЫ МЕЖДУ non И full', b2_cohort_stability(), 'Если поломка меняла сам ОТБОР в когорту, когортный анализ non несопоставим с full.')
    show('C. МЕХАНИЗМ: НЕПОЛНОЕ ГОЛОСОВАНИЕ В R1 И ЕГО ЦЕНА', b3_mechanism(), 'protocol_n_valid_votes < k означает, что голос эксперта не дошёл до организатора.')
    for v in VARIANTS:
        show('D. ПОДГРУППЫ ПО Type ВНУТРИ C2+C3 (§4 исходного документа) — %s' % v.upper(), b4_type(v).set_index(['плечо', 'Type']))
    for v in VARIANTS:
        show('F. РЕГРЕССИЯ К СЕРЕДИНЕ: КВАРТИЛИ ПО КАЧЕСТВУ R1 (§5) — %s' % v.upper(), b5_regression_to_mean(v).set_index(['плечо', 'квартиль']))
    for v in VARIANTS:
        show('G. СТАДИЯ ПРОТИВ СОБСТВЕННОЙ ОДИНОЧНОЙ МОДЕЛИ — %s' % v.upper(), b6_vs_single(v).set_index(['плечо', 'когорта', 'стадия']), 'Проверяется тезис «при обрезке на R1 плечо падает ниже своей одиночной модели».')
    for v in VARIANTS:
        main_t, inter_t, p_inter = b7_pooled(v)
        show('H. ОБЪЕДИНЁННАЯ МОДЕЛЬ ПО ВСЕМ ПЛЕЧАМ, C2+C3 — %s' % v.upper(), main_t, 'Случайный эффект кейса; is_fin = эффект доведения до финала против остановки на R1.')
        show('H2. ВЗАИМОДЕЙСТВИЕ СТАДИЯ x ПЛЕЧО — %s (LR-тест p = %.4g)' % (v.upper(), p_inter), inter_t)
    show('H3. МЕДИАЦИЯ: ОБЪЯСНЯЕТСЯ ЛИ ЭФФЕКТ РАУНДОВ ПОТЕРЕЙ ГОЛОСОВ В R1', b7b_mediation())
    for v in VARIANTS:
        show('I. ЭФФЕКТ РАУНДОВ ПО ОСЯМ РУБРИКИ, C2+C3 — %s' % v.upper(), b8_axes(v).set_index(['плечо', 'ось']))
    for v in VARIANTS:
        show('J. ЭКВИВАЛЕНТНОСТЬ (TOST) И МОЩНОСТЬ, C2+C3 — %s' % v.upper(), b9_equivalence(v), 'TOST < 0.05 => доказано ОТСУТСТВИЕ эффекта больше границы. MDE -- что вообще видно при этом n.')
    per_fam, tot, reg = b10_multiplicity()
    show('K. МНОЖЕСТВЕННОСТЬ ПО СЕМЕЙСТВАМ', per_fam)
    show('K2. ИТОГО ПО ВАРИАНТУ', tot)
    reg.to_csv(os.path.normpath(os.path.join(OUT, 'round_effect_contrasts.csv')), index=False, encoding='utf-8-sig')
    per_fam.to_csv(os.path.normpath(os.path.join(OUT, 'multiple_comparison_results_by_test_family.csv')), encoding='utf-8-sig')
    piv = reg.pivot_table(index=['family', 'arm', 'contrast'], columns='variant', values='p', aggfunc='first')
    if set(VARIANTS).issubset(piv.columns):
        piv['значим только non'] = (piv['non'] < ALPHA) & ~(piv['full'] < ALPHA)
        piv['значим только full'] = ~(piv['non'] < ALPHA) & (piv['full'] < ALPHA)
        flip = piv[piv['значим только non'] | piv['значим только full']]
        show('L. КОНТРАСТЫ, У КОТОРЫХ ЗНАЧИМОСТЬ ПОЯВИЛАСЬ ИЛИ ИСЧЕЗЛА', flip)
        piv.to_csv(os.path.normpath(os.path.join(OUT, 'significance_changes_after_truncation_correction.csv')), encoding='utf-8-sig')
    print('\n' + '=' * 104)
    print('CSV: %s' % OUT)
    print('=' * 104)
if __name__ == '__main__':
    main()


УГЛУБЛЁННАЯ ПРОВЕРКА ЭФФЕКТА РАУНДОВ + ИНВЕНТАРИЗАЦИЯ ЗНАЧИМОСТИ
Automatically created module for IPython interactive environment



A. КОГОРТНЫЙ ДИЗАЙН (воспроизведение analysis_rounds_vs_final §2) — NON
n≠0 -- сколько кейсов дали НЕнулевую разность; у C1 их ноль по построению.
                            n  n≠0        R1       FIN    FIN−R1     доля>  p(FIN~R1)        R2    FIN−R2  p(FIN~R2)
плечо            когорта                                                                                            
nofix_original   C1       134    0  0.767472  0.767472  0.000000  0.000000        NaN  0.767472  0.000000        NaN
                 C2        53   32  0.777132  0.796053  0.018921  0.339623   0.261889  0.796053  0.000000        NaN
                 C3        37   26  0.749389  0.787384  0.037995  0.486486   0.030857  0.776773  0.010611   0.511089
                 C2+C3     90   58  0.765727  0.792489  0.026762  0.400000   0.021920  0.788127  0.004362   0.511089
                 все      224   58  0.766771  0.777523  0.010753  0.160714   0.021920  0.775771  0.001753   0.511089
fix_original     C1       108    


A. КОГОРТНЫЙ ДИЗАЙН (воспроизведение analysis_rounds_vs_final §2) — FULL
n≠0 -- сколько кейсов дали НЕнулевую разность; у C1 их ноль по построению.
                            n  n≠0        R1       FIN    FIN−R1     доля>  p(FIN~R1)        R2    FIN−R2  p(FIN~R2)
плечо            когорта                                                                                            
nofix_original   C1       120    0  0.778497  0.778497  0.000000  0.000000        NaN  0.778497  0.000000        NaN
                 C2        62   45  0.798587  0.817811  0.019224  0.435484   0.190407  0.817811  0.000000        NaN
                 C3        47   40  0.777440  0.782421  0.004981  0.446809   0.882450  0.819038 -0.036617   0.027330
                 C2+C3    109   85  0.789469  0.802551  0.013083  0.440367   0.400175  0.818340 -0.015789   0.027330
                 все      229   85  0.783719  0.789946  0.006227  0.209607   0.400175  0.797462 -0.007515   0.027330
fix_original     C1       108   


B. УСТОЙЧИВОСТЬ СОСТАВА «ТРУДНОЙ» КОГОРТЫ МЕЖДУ non И full
Если поломка меняла сам ОТБОР в когорту, когортный анализ non несопоставим с full.
                    n  трудных non  трудных full  в обоих  только non  только full       phi  p (McNemar)
плечо                                                                                                    
nofix_original    224           90           105       72          18           33  0.544007     0.048874
homog_deepseekv4  229           59            58       25          34           33  0.230904     1.000000



C. МЕХАНИЗМ: НЕПОЛНОЕ ГОЛОСОВАНИЕ В R1 И ЕГО ЦЕНА
protocol_n_valid_votes < k означает, что голос эксперта не дошёл до организатора.
                            n  доля полных голосований R1  R1 при полном  R1 при неполном   p (MWU)  доля трудных при полном  доля трудных при неполном
вариант плечо                                                                                                                                          
non     homog_deepseekv4  229                    0.358079       0.691168         0.669035  0.522326                 0.146341                   0.319728
        homog_gpt         229                    0.834061       0.753365         0.777989  0.408909                 0.246073                   0.236842
        pmixed            229                    0.995633       0.794485         0.455900  0.100597                 0.381579                   0.000000
full    nofix_original    229                    0.991266       0.782474         0.925000  0.112516        

...


I. ЭФФЕКТ РАУНДОВ ПО ОСЯМ РУБРИКИ, C2+C3 — NON
                                         n        R1       FIN    FIN−R1     доля>         p
плечо            ось                                                                        
homog_deepseekv4 reasoning              45  0.708561  0.765883  0.057321  0.222222  0.546390
                 accuracy               58  0.777591  0.849054  0.071464  0.293103  0.033501
                 completeness           58  0.397343  0.539524  0.142181  0.448276  0.000967
                 urgency                59  0.619456  0.770238  0.150783  0.271186  0.107528
                 uncertainty            50  0.367576  0.502695  0.135119  0.260000  0.052327
                 communication_quality  45  0.651389  0.595833 -0.055556  0.000000  0.102470
nofix_original   reasoning              72  0.856798  0.892107  0.035308  0.069444  0.250666
                 accuracy               89  0.851765  0.873474  0.021709  0.101124  0.432768
                 compl


I. ЭФФЕКТ РАУНДОВ ПО ОСЯМ РУБРИКИ, C2+C3 — FULL
                                          n        R1       FIN    FIN−R1     доля>         p
плечо            ось                                                                         
homog_deepseekv4 reasoning               41  0.945122  0.935976 -0.009146  0.024390  0.785495
                 accuracy                58  0.856430  0.807323 -0.049106  0.051724  0.092389
                 completeness            57  0.629704  0.600969 -0.028735  0.105263  0.324117
                 urgency                 58  0.950770  0.971706  0.020936  0.034483  0.179712
                 uncertainty             47  0.651006  0.643028 -0.007979  0.042553  0.915583
                 communication_quality   45  0.703704  0.756481  0.052778  0.088889  0.257899
nofix_original   reasoning               86  0.937177  0.928559 -0.008618  0.034884  0.672052
                 accuracy               108  0.899674  0.895946 -0.003728  0.101852  0.909531
           


J. ЭКВИВАЛЕНТНОСТЬ (TOST) И МОЩНОСТЬ, C2+C3 — NON
TOST < 0.05 => доказано ОТСУТСТВИЕ эффекта больше границы. MDE -- что вообще видно при этом n.
                    n    FIN−R1  p (Уилкоксон)  TOST ±0.05    TOST ±0.10  MDE (80%)
плечо                                                                              
nofix_original     90  0.026762       0.021920    0.028176  1.382059e-08   0.033671
fix_original      121  0.026583       0.027006    0.013693  7.851410e-11   0.029377
pstrong           101  0.008035       0.427319    0.001997  1.932186e-09   0.039895
pweak             110 -0.004174       0.834773    0.000039  3.500957e-14   0.031266
pmixed             87 -0.017380       0.221159    0.013235  7.593756e-08   0.040471
homog_gpt          56  0.005366       0.927875    0.007862  1.113517e-06   0.050168
homog_gemini       37  0.009295       0.943054    0.025394  3.392193e-05   0.056433
homog_deepseekv4   59  0.112205       0.000448    0.981808  6.620985e-01   0.081351



J. ЭКВИВАЛЕНТНОСТЬ (TOST) И МОЩНОСТЬ, C2+C3 — FULL
TOST < 0.05 => доказано ОТСУТСТВИЕ эффекта больше границы. MDE -- что вообще видно при этом n.
                    n    FIN−R1  p (Уилкоксон)  TOST ±0.05    TOST ±0.10  MDE (80%)
плечо                                                                              
nofix_original    109  0.013083       0.400175    0.000531  1.122755e-12   0.030738
fix_original      121  0.026583       0.027006    0.013693  7.851410e-11   0.029377
pstrong           101  0.008035       0.427319    0.001997  1.932186e-09   0.039895
pweak             110 -0.004174       0.834773    0.000039  3.500957e-14   0.031266
pmixed             87 -0.017380       0.221159    0.013235  7.593756e-08   0.040471
homog_gpt          56  0.005366       0.927875    0.007862  1.113517e-06   0.050168
homog_gemini       37  0.009295       0.943054    0.025394  3.392193e-05   0.056433
homog_deepseekv4   58 -0.018172       0.085754    0.002267  1.600000e-10   0.030170



K. МНОЖЕСТВЕННОСТЬ ПО СЕМЕЙСТВАМ
                                    тестов  p<0.05 сырых  Holm  BH
семейство                  вариант                                
A. когорты FIN−R1          full         16             2     1   1
                           non          16             5     2   2
B. шаги внутри C3          full         16             5     1   1
                           non          16             3     1   1
C. механизм: неполнота R1  full          4             0     0   0
                           non           3             0     0   0
D. Type-подгруппы @C2+C3   full         24             2     0   0
                           non          24             5     1   1
E. Kruskal-Wallis по Type  full          8             0     0   0
                           non           8             1     1   1
F. регрессия к середине    full          8             6     5   5
                           non           8             6     6   6
G. стадия против одиночной f